# Minimal MLP Regression Sanity Check

Toy experiment, not a thesis result. Purpose: verify that a small
PyTorch training loop (data -> model -> loss -> optimizer -> plot)
runs correctly end to end in a fresh Colab runtime.

**Data.** $y = 3x + 2 + \varepsilon$, $\varepsilon \sim \mathcal{N}(0, \sigma^2)$, $N = 1000$ samples.

**Model.** `Linear(1, 32) -> ReLU -> Linear(32, 1)`

**Loss / optimizer.** `MSELoss`, `Adam`.

This notebook is fully self-contained (no repo clone, no extra
dependencies) since `torch` and `matplotlib` are preinstalled in Colab.

## 1. GPU check and imports

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

## 2. Generate synthetic data

$x \in \mathbb{R}^{N \times 1}$, $y \in \mathbb{R}^{N \times 1}$.
The trailing dimension of size 1 is the single input/output feature,
matching the `Linear(1, 32)` layer, which expects input shape `[B, 1]`.

In [ ]:
N = 1000
true_slope = 3.0
true_intercept = 2.0
noise_std = 1.0

# x: [N] -> [N, 1] so it matches the model's expected input shape [B, 1]
x = torch.linspace(-5.0, 5.0, N).unsqueeze(1)
noise = torch.randn(N, 1) * noise_std          # [N, 1]
y = true_slope * x + true_intercept + noise    # [N, 1]

print(f"x shape = {tuple(x.shape)}, y shape = {tuple(y.shape)}")

## 3. Model, loss, optimizer

Forward shapes: `[B, 1] -> Linear(1,32) -> [B, 32] -> ReLU -> [B, 32]
-> Linear(32,1) -> [B, 1]`.

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

## 3b. Tensor shape check (single forward pass)

One-off, untrained forward pass on a small batch, printing the tensor
shape after every layer. This is purely for shape verification and is
not part of training.

In [ ]:
sample_batch = x[:4].to(device)  # [4, 1] -- 4 samples, 1 input feature
print(f"input                : {tuple(sample_batch.shape)}")

h = sample_batch
for layer in model:
    h = layer(h)
    print(f"after {layer.__class__.__name__:<15}: {tuple(h.shape)}")

## 4. Training loop

Full-batch gradient descent (all 1000 points per step, no DataLoader)
since the dataset is tiny -- this keeps the loop explicit and simple.

In [ ]:
x_dev = x.to(device)
y_dev = y.to(device)

n_epochs = 500

for epoch in range(n_epochs):
    optimizer.zero_grad()

    y_pred = model(x_dev)          # [N, 1]
    loss = loss_fn(y_pred, y_dev)  # scalar

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"epoch {epoch + 1:4d}/{n_epochs}  mse_loss = {loss.item():.4f}")

## 5. Plot data and learned function

In [ ]:
model.eval()
with torch.no_grad():
    y_learned = model(x_dev).cpu()  # [N, 1]

x_np = x.numpy()
y_np = y.numpy()
y_learned_np = y_learned.numpy()

plt.figure(figsize=(7, 5))
plt.scatter(x_np, y_np, s=8, alpha=0.4, label="noisy data")
plt.plot(x_np, y_learned_np, color="red", linewidth=2, label="learned function")
plt.plot(
    x_np, true_slope * x_np + true_intercept,
    color="black", linestyle="--", linewidth=1, label="true line (no noise)",
)
plt.xlabel("x")
plt.ylabel("y")
plt.title("MLP regression: y = 3x + 2 + noise")
plt.legend()
plt.show()